In [12]:
import pandas as pd
import os
import json

dataset_id = "Metanova/SAVI-2020"
dataset_config = "default"
configs_file = "config.json"
compression_methods = ["snappy", "gzip", "brotli", "lz4", "zstd"]
config_dict = json.load(open(configs_file, "r"))[dataset_id]

In [25]:
def get_df(path, max_rows=1_000_000):
    parts = []
    total_rows = 0

    for fname in sorted(os.listdir(path)):
        df_part = pd.read_parquet(os.path.join(path, fname))
        n = len(df_part)

        if total_rows + n >= max_rows:
            # Only take what we still need to reach the cap
            df_part = df_part.iloc[: max_rows - total_rows]
            parts.append(df_part)
            break

        parts.append(df_part)
        total_rows += n

        if total_rows >= max_rows:
            break

    return pd.concat(parts, ignore_index=True)

def compare_dataframes(df1, df2, ignore_columns=None):
    if ignore_columns is None:
        ignore_columns = []

    # Drop ignored columns from both DataFrames
    df1 = df1.drop(columns=[col for col in ignore_columns if col in df1.columns])
    df2 = df2.drop(columns=[col for col in ignore_columns if col in df2.columns])

    # Align columns by name
    df1_sorted = df1.sort_index(axis=1)
    df2_sorted = df2[df1_sorted.columns]  # reorder df2 to match df1

    # Check shape first
    if df1_sorted.shape != df2_sorted.shape:
        print("DataFrames have different shapes:", df1_sorted.shape, df2_sorted.shape)
        return

    # Create a boolean DataFrame of element-wise comparisons
    diff = df1_sorted != df2_sorted

    if not diff.any().any():
        print("DataFrames are equal")
    else:
        print("Differences found at these locations:")
        differing_cells = diff.stack()[diff.stack()]  # True where differences exist
        for (idx, col) in differing_cells.index:
            print(f"- Row {idx}, Column '{col}': df1 = {df1_sorted.at[idx, col]!r}, df2 = {df2_sorted.at[idx, col]!r}")


def get_size(start_path = '.'):
    # check is it a file or directory
    if os.path.isfile(start_path):
        return os.path.getsize(start_path)
    total = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

In [14]:
df = get_df(config_dict['configs'][0]['path'])
print(df.head())

                                  product_name     product_hashisy  \
0  "61A77BD3826E0AD0_554BFAE008D564C8_6031_UN"  "61A77BD3826E0AD0"   
1  "46BED227B506FAC0_C42C347BD57866CD_6031_UN"  "46BED227B506FAC0"   
2  "D5A12E82DD1700C8_C2748EE6925A3706_6031_UN"  "D5A12E82DD1700C8"   
3  "0D478AE9F05191D4_9B5CA2C45361352F_6031_UN"  "0D478AE9F05191D4"   
4  "6A4C62355217EA6C_D39599117C2C621E_6031_UN"  "6A4C62355217EA6C"   

                                      product_smiles         formula  \
0           "O(C(=O)C1=C(OC=N1)C2=CC=CS2)CCC3OCCC3C"    "C15H17NO4S"   
1  "O(C(=O)C1=C(OC=N1)C2=CC=CS2)C[C@H]4[C@@H]3C[C...   "C16H18N2O3S"   
2  "O(C(=O)C1=C(OC=N1)C2=CC=CS2)CC4OCCN(C3=NC=C[N...   "C17H18N4O4S"   
3     "O(C(=O)C1=C(OC=N1)C2=CC=CS2)CC3=NC(=CC=N3)Cl"  "C13H8ClN3O3S"   
4  "O(C(=O)C1=C(OC=N1)C2=CC=CS2)CC3=CC=C(C=C3)C[N...   "C23H17N3O3S"   

     weight  rotbonds  h_donors  h_acceptors  e_tpsa    fsp3  ...  \
0  307.3636         6         0            6   89.80  0.4667  ...   
1  318.3

In [15]:
os.makedirs("datasets_compress/Metanova/SAVI-2020/", exist_ok=True)

In [16]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
def smiles_to_formula(smiles):
    mol = Chem.MolFromSmiles(smiles.replace('"', ''))
    if mol is None:
        return None
    return rdMolDescriptors.CalcMolFormula(mol)

In [19]:
compress_df = df.copy()
# compress_df = compress_df[["product_smiles", "formula"]]
# compress_df["computed_formula"] = compress_df["product_smiles"].apply(smiles_to_formula)
compress_df = compress_df.drop(columns=["formula"])
print(compress_df.head())

                                  product_name     product_hashisy  \
0  "61A77BD3826E0AD0_554BFAE008D564C8_6031_UN"  "61A77BD3826E0AD0"   
1  "46BED227B506FAC0_C42C347BD57866CD_6031_UN"  "46BED227B506FAC0"   
2  "D5A12E82DD1700C8_C2748EE6925A3706_6031_UN"  "D5A12E82DD1700C8"   
3  "0D478AE9F05191D4_9B5CA2C45361352F_6031_UN"  "0D478AE9F05191D4"   
4  "6A4C62355217EA6C_D39599117C2C621E_6031_UN"  "6A4C62355217EA6C"   

                                      product_smiles    weight  rotbonds  \
0           "O(C(=O)C1=C(OC=N1)C2=CC=CS2)CCC3OCCC3C"  307.3636         6   
1  "O(C(=O)C1=C(OC=N1)C2=CC=CS2)C[C@H]4[C@@H]3C[C...  318.3898         5   
2  "O(C(=O)C1=C(OC=N1)C2=CC=CS2)CC4OCCN(C3=NC=C[N...  374.4136         6   
3     "O(C(=O)C1=C(OC=N1)C2=CC=CS2)CC3=NC(=CC=N3)Cl"  321.7375         5   
4  "O(C(=O)C1=C(OC=N1)C2=CC=CS2)CC3=CC=C(C=C3)C[N...  415.4656         7   

   h_donors  h_acceptors  e_tpsa    fsp3  complexity  ...    r2_bbsource  \
0         0            6   89.80  0.4667    36

In [18]:
# compate the computed formula with the original formula
compress_df["computed_formula"] = compress_df["computed_formula"].apply(lambda x: '"' + x + '"')
compress_df["prediction_correct"] = compress_df["formula"] == compress_df["computed_formula"]
mean_prediction_correct = compress_df["prediction_correct"].mean()
print(f"Mean prediction correct: {mean_prediction_correct:.2%}")

Mean prediction correct: 99.87%


In [20]:
import pyarrow as pa
import pyarrow.parquet as pq

def write_parquet(df, path, PARQUET_COMPRESSION_TYPE="snappy"):
    table = pa.Table.from_pandas(df)
    pq.write_table(table, path, compression=PARQUET_COMPRESSION_TYPE)

write_parquet(compress_df, f"datasets_compress/{dataset_id}/{dataset_config}.parquet")

In [21]:
compress_df = pd.read_parquet(f"datasets_compress/{dataset_id}/{dataset_config}.parquet")
print(compress_df.info())
print(compress_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 71 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   product_name                      1000000 non-null  object 
 1   product_hashisy                   1000000 non-null  object 
 2   product_smiles                    1000000 non-null  object 
 3   weight                            1000000 non-null  float64
 4   rotbonds                          1000000 non-null  int64  
 5   h_donors                          1000000 non-null  int64  
 6   h_acceptors                       1000000 non-null  int64  
 7   e_tpsa                            1000000 non-null  float64
 8   fsp3                              1000000 non-null  float64
 9   complexity                        1000000 non-null  float64
 10  product_qed_score                 991793 non-null   float64
 11  benzenoid_index                   1000

In [22]:
size = config_dict['configs'][0]['size_snappy']
compression_size = get_size(f"datasets_compress/{dataset_id}/{dataset_config}.parquet")
print(f"Size of the dataset: {size / (1024):.2f} kB")
print(f"Size of the dataset: {size} B")
print(f"Size of the compression: {compression_size / (1024):.2f} kB")
print(f"Size of the compression: {compression_size} B")
print(f"Compression ratio: {((size-compression_size) / size) * 100:.2f} %")

Size of the dataset: 465265.54 kB
Size of the dataset: 476431909 B
Size of the compression: 460685.10 kB
Size of the compression: 471741539 B
Compression ratio: 0.98 %


In [23]:
# compress_df['product_name'] = compress_df['product_name'].str[0] + compress_df['product_hashisy'].str[1:-1] + compress_df['product_name'].str[1:]
# compress_df['r2_ident'] = compress_df['r2_url'].apply(lambda x: '"' + x.split("/")[-1])
# compress_df['r1_ident'] = compress_df['r1_url'].apply(lambda x: '"' + x.split("/")[-1])
# compress_df['r1_url'] = compress_df['r1_url'].str[:-1] + compress_df['r1_ident'].str[1:-1] + compress_df['r1_url'].str[-1]
# compress_df['r2_url'] = compress_df['r2_url'].str[:-1] + compress_df['r2_ident'].str[1:-1] + compress_df['r2_url'].str[-1]
compress_df["formula"] = compress_df["product_smiles"].apply(smiles_to_formula)
compress_df["formula"] = compress_df["formula"].apply(lambda x: '"' + x + '"')

In [27]:
compare_dataframes(df, compress_df, ignore_columns=["ro5_violations", "ro3_violations", "xlogp", "xlogp2", "product_qed_score"])

Differences found at these locations:
- Row 530, Column 'formula': df1 = '"C18H19N2O5S2"', df2 = '"C18H19N2O5S2+"'
- Row 1344, Column 'formula': df1 = '"C13H24N3O3"', df2 = '"C13H24N3O3+"'
- Row 2086, Column 'formula': df1 = '"C28H54N3O3"', df2 = '"C28H54N3O3+"'
- Row 3313, Column 'formula': df1 = '"C20H27N6O3S"', df2 = '"C20H27N6O3S+"'
- Row 3532, Column 'formula': df1 = '"C16H22N3O3"', df2 = '"C16H22N3O3+"'
- Row 3725, Column 'formula': df1 = '"C16H30N3O3"', df2 = '"C16H30N3O3+"'
- Row 4193, Column 'formula': df1 = '"C26H36N3O5"', df2 = '"C26H36N3O5+"'
- Row 5888, Column 'formula': df1 = '"C18H26N3O5S"', df2 = '"C18H26N3O5S+"'
- Row 6702, Column 'formula': df1 = '"C17H23N4O4"', df2 = '"C17H23N4O4+"'
- Row 7444, Column 'formula': df1 = '"C32H53N4O4"', df2 = '"C32H53N4O4+"'
- Row 8671, Column 'formula': df1 = '"C24H26N7O4S"', df2 = '"C24H26N7O4S+"'
- Row 8890, Column 'formula': df1 = '"C20H21N4O4"', df2 = '"C20H21N4O4+"'
- Row 9083, Column 'formula': df1 = '"C20H29N4O4"', df2 = '"C20H2